# CropGym tutorial: nitrogen management with WOFOST SNOMIN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WUR-AI/PCSE-Gym/blob/feature/pcse6-wofost-snomin-tutorial/notebooks/tutorials/CropGym_WOFOST_SNOMIN_Tutorial.ipynb)

This notebook is **self-contained for Google Colab**: run cells top-to-bottom. It installs CropGym, PCSE 6.x, and Stable-Baselines3 automatically.

You will:
1. Configure a CropGym environment (`pcse_model=2`, WOFOST + SNOMIN)
2. Inspect observations (including layered soil N)
3. Compare baseline fertilization policies
4. Train a short PPO agent

Runtime: ~15–30 min on Colab CPU (mostly NASA weather fetch + short PPO training).

> **Colab:** run the install cell once — it restarts the runtime automatically. Then run the install cell again (fast), then continue with the cells below.

> Pip may warn about other Colab packages (jax, opencv, etc.) — safe to ignore; this tutorial does not use them.

> PCSE may print **traitlets DeprecationWarnings** when starting a simulation — these are harmless upstream warnings and do not affect results.

## 1. Install CropGym (Colab)

In [ ]:
#@title Install dependencies and clone CropGym { display-mode: "form" }
import sys
import subprocess
import os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/WUR-AI/PCSE-Gym.git'
REPO_BRANCH = 'feature/pcse6-wofost-snomin-tutorial'
SETUP_VERSION = '2'
repo_root = Path('/content/PCSE-Gym') if IN_COLAB else Path.cwd()
setup_marker = repo_root / f'.colab_setup_v{SETUP_VERSION}_done'

if IN_COLAB:
    if not repo_root.exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, str(repo_root)],
            check=True,
        )
    os.chdir(repo_root)

    if not setup_marker.exists():
        pip = [sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir']
        subprocess.run([*pip, '--upgrade', 'pip'], check=True)
        # Match Colab's NumPy 2.x stack to avoid ABI mismatch and version warnings
        subprocess.run(
            [*pip, '--force-reinstall', 'numpy>=2.0,<3', 'scipy', 'pandas'],
            check=True,
        )
        deps = [
            'pcse>=6.0.13', 'gymnasium>=0.29', 'pyyaml',
            'stable-baselines3>=2.3', 'sb3-contrib', 'matplotlib',
            'tensorboard', 'tqdm', 'seaborn',
        ]
        subprocess.run([*pip, *deps], check=True)
        # --no-deps: pyproject pins numpy<2 for local Poetry, but Colab needs numpy 2.x
        subprocess.run([*pip, '-e', str(repo_root), '--no-deps'], check=True)
        setup_marker.write_text('ok')
        print('Install complete — restarting runtime (required for NumPy)...')
        os.kill(os.getpid(), 9)
    else:
        print('Dependencies installed — setting up paths')
else:
    candidate = Path.cwd().resolve()
    for _ in range(4):
        if (candidate / 'pcse_gym').exists():
            repo_root = candidate
            break
        candidate = candidate.parent
    print('Local mode — ensure: poetry install --extras sb-integration')

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Writable PCSE cache/logs (must be set before importing pcse)
pcse_home = repo_root / '.pcse_home'
pcse_home.mkdir(exist_ok=True)
(pcse_home / 'logs').mkdir(exist_ok=True)
(pcse_home / 'meteo_cache').mkdir(exist_ok=True)
os.environ['HOME'] = str(pcse_home)

print('repo_root:', repo_root)
print('Python:', sys.version.split()[0])

In [ ]:
#@title Verify installation { display-mode: "form" }
import os
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    repo_root = Path('/content/PCSE-Gym')
    pcse_home = repo_root / '.pcse_home'
    os.environ['HOME'] = str(pcse_home)
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

import numpy as np
import pcse
import gymnasium
import stable_baselines3

from pcse_gym.envs.sb3 import get_config_dir

config_dir = Path(get_config_dir())
required = [
    config_dir / 'Wofost81_NWLP_MLWB_SNOMIN.conf',
    config_dir / 'soil' / 'arminda_soil.yaml',
    config_dir / 'site' / 'arminda_site.yaml',
    config_dir / 'agro' / 'wheat_cropcalendar_snomin.yaml',
    config_dir / 'crop' / 'winterwheat.yaml',
]
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing SNOMIN config files:\n' + '\n'.join(map(str, missing)))

print('numpy', np.__version__)
print('pcse', getattr(pcse, '__version__', 'installed'))
print('gymnasium', gymnasium.__version__)
print('stable-baselines3', stable_baselines3.__version__)
print('SNOMIN configs OK')

## 2. Imports and settings

In [ ]:
import os
import warnings

# PCSE 6.x + Colab traitlets: noisy but harmless DeprecationWarnings
warnings.filterwarnings('ignore', category=DeprecationWarning, module='pcse')

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import gymnasium as gym

from pcse_gym.envs.winterwheat import WinterWheat
from pcse_gym.envs.sb3 import get_model_kwargs, get_policy_kwargs
from pcse_gym.utils.eval import evaluate_policy
from pcse_gym.utils.nitrogen_helpers import get_nitrogen_levels
import pcse_gym.utils.defaults as defaults

PCSE_MODEL = 2  # WOFOST 8.1 + SNOMIN
crop_features = defaults.get_snomin_default_crop_features()
weather_features = defaults.get_default_weather_features()
action_features = defaults.get_default_action_features()
action_space = defaults.get_snomin_action_space()
nitrogen_levels = get_nitrogen_levels()
n_levels = len(nitrogen_levels)
costs_nitrogen = 10.0

print('Crop features:', crop_features)
print('Nitrogen levels (kg N/ha):', nitrogen_levels)

## 3. Create the CropGym environment

`WinterWheat` wraps PCSE in a Gymnasium interface. With `pcse_model=2` it uses:
- `Wofost81_NWLP_MLWB_SNOMIN.conf`
- multi-layer soil files (`arminda_soil.yaml`, `arminda_site.yaml`)
- winter wheat variety **Arminda** (`wheat_cropcalendar_snomin.yaml`)

The default reward (`DEF`) compares yield growth against a zero-nitrogen baseline, minus an economic cost for fertilizer.

In [ ]:
def make_env(years, locations, seed=0):
    return WinterWheat(
        crop_features=crop_features,
        action_features=action_features,
        weather_features=weather_features,
        costs_nitrogen=costs_nitrogen,
        years=years,
        locations=locations,
        action_space=action_space,
        reward='DEF',
        seed=seed,
        n_nitrogen_levels=n_levels,
        **get_model_kwargs(PCSE_MODEL),
    )

demo_year = 2001
demo_location = (52, 5.5)
env = make_env(demo_year, demo_location)
obs, info = env.reset()
print('Observation shape:', obs.shape)
print('Action space:', env.action_space)
print('Observation space:', env.observation_space)

## 4. Run one growing season manually

Actions are discrete indices mapped to fertilizer rates in **kg N/ha**. The agent decides weekly (7-day timesteps).

In [ ]:
env = make_env(demo_year, demo_location, seed=1)
obs, _ = env.reset()
total_reward = 0.0
history = []
terminated = truncated = False
while not (terminated or truncated):
    action = int(env.action_space.sample())
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    history.append({
        'day': env.date,
        'action_idx': action,
        'fertilizer_kg': nitrogen_levels[action],
        'reward': reward,
        'WSO': info.get('WSO', {}).get(env.date, np.nan),
        'NLOSSCUM': info.get('NLOSSCUM', {}).get(env.date, np.nan),
    })

df_run = pd.DataFrame(history)
print(f'Total reward: {total_reward:.1f}')
df_run.tail()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
df_run.set_index('day')['WSO'].plot(ax=axes[0], title='Storage organ weight (WSO, kg/ha)')
df_run.set_index('day')['NLOSSCUM'].plot(ax=axes[1], title='Cumulative N loss (NLOSSCUM)')
df_run.set_index('day')['fertilizer_kg'].plot(ax=axes[2], drawstyle='steps-post', title='Applied N (kg/ha)')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 5. Compare baseline policies

In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

def run_policy(policy, year, location):
    env = make_env(year, location)
    env_vec = VecNormalize(
        DummyVecEnv([lambda y=year, loc=location: make_env(y, loc)]),
        norm_reward=True, clip_reward=50.0, gamma=1,
    )
    env_vec.training = False
    env_vec.norm_reward = True
    rewards, infos = evaluate_policy(policy, env_vec)
    return rewards[0], infos[0]

year, location = 2000, (52, 5.5)
policies = {
    'zero-N': 'no-nitrogen',
    'standard practice': 'standard-practice',
}
for name, policy in policies.items():
    reward, info = run_policy(policy, year, location)
    wso = info['WSO']
    final_wso = float(list(wso.values())[-1])
    total_n = float(sum(info['fertilizer'].values()))
    print(f'{name:20s} reward={float(reward):8.1f}  final WSO={final_wso:8.1f} kg/ha  total N={total_n:.1f} kg/ha')

## 6. Train a PPO agent (short demo)

Training uses CPU by default (stable on free Colab). Increase `nsteps` for better policies (e.g. 50_000).

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from pcse_gym.utils.eval import EvalCallback

train_years = [1999, 2001, 2003]
test_years = [2000, 2002]
train_locations = [(52, 5.5)]
test_locations = [(52, 5.5)]
nsteps = 5_000  # Colab demo; use 50_000+ for real experiments
seed = 0

env_train = make_env(train_years, train_locations, seed=seed)
env_train = Monitor(env_train)
env_train = VecNormalize(
    DummyVecEnv([lambda: make_env(train_years, train_locations, seed=seed)]),
    norm_obs=True, norm_reward=True, clip_obs=10.0, clip_reward=50.0, gamma=1,
)
env_eval = make_env(test_years, test_locations, seed=seed)

hyperparams = {
    'batch_size': 64,
    'n_steps': 2048,
    'learning_rate': 3e-4,
    'clip_range': 0.3,
    'policy_kwargs': get_policy_kwargs(
        n_crop_features=len(crop_features),
        n_weather_features=len(weather_features),
        n_action_features=len(action_features),
    ),
    'device': 'cpu',
}
hyperparams['policy_kwargs']['net_arch'] = dict(pi=[128, 128], vf=[128, 128])

log_dir = repo_root / 'notebooks' / 'tutorials' / 'tensorboard_snomin'
log_dir.mkdir(parents=True, exist_ok=True)

model = PPO('MlpPolicy', env_train, gamma=1, seed=seed, verbose=1,
            tensorboard_log=str(log_dir), **hyperparams)
model.learn(
    total_timesteps=nsteps,
    callback=EvalCallback(
        env_eval=env_eval,
        test_years=test_years,
        train_years=train_years,
        train_locations=train_locations,
        test_locations=test_locations,
        eval_freq=nsteps,
        pcse_model=PCSE_MODEL,
    ),
    tb_log_name='SNOMIN-PPO-demo',
)

In [ ]:
# Optional: view TensorBoard in Colab
try:
    get_ipython().run_line_magic('load_ext', 'tensorboard')
    get_ipython().run_line_magic('tensorboard', f'--logdir {log_dir}')
except Exception as exc:
    print('TensorBoard not available:', exc)

## 7. Evaluate the trained agent

In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv

eval_env = DummyVecEnv([lambda: make_env(2002, (52, 5.5))])
rewards, infos = evaluate_policy(model, eval_env)
info = infos[0]
print(f'RL reward: {float(rewards[0]):.1f}')
print(f'Final WSO: {float(list(info["WSO"].values())[-1]):.1f} kg/ha')
print(f'Total N applied: {float(sum(info["fertilizer"].values())):.1f} kg/ha')
print(f'Cumulative N loss: {float(list(info["NLOSSCUM"].values())[-1]):.1f}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
pd.Series(info['WSO']).plot(ax=ax[0], title='WSO under learned policy')
pd.Series(info['NLOSSCUM']).plot(ax=ax[1], title='Cumulative N loss')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 8. Next steps

- Increase `nsteps` for better policies
- Full CLI training: `python train_winterwheat.py --environment 2 --agent PPO --nsteps 50000`
- NUE-focused rewards and constrained RL: [NUE_PCSE-Gym](https://github.com/WUR-AI/NUE_PCSE-Gym)
- Simpler LINTUL-3 tutorial: `Advanced_Machine_Learning_CropGym_Tutorial.ipynb`